In [2]:
import os, sys
sys.path.append(os.path.abspath('..'))
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from scripts.model import LSTMModel, GRUModel

df = pd.read_csv("../data/processed/AMZN_clean_prices.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
prices = df['Price'].values.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(-1, 1))
scaled_prices = scaler.fit_transform(prices)

In [ ]:
def create_sliding_windows(data, lookback=20):
    x, y = [], []
    for i in range(len(data) - lookback):
        x.append(data[i:i + lookback])
        y.append(data[i + lookback])
    return np.array(x), np.array(y)

lookback = 20
X, y = create_sliding_windows(scaled_prices, lookback)

train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
lstm_model = LSTMModel(input_dim=1, hidden_dim=64, num_layers=2, output_dim=1)
gru_model = GRUModel(input_dim=1, hidden_dim=64, num_layers=2, output_dim=1)

criterion = torch.nn.MSELoss()
optimizer_lstm = torch.optim.Adam(lstm_model.parameters(), lr=0.01)
optimizer_gru = torch.optim.Adam(gru_model.parameters(), lr=0.01)

In [ ]:
epochs = 60
for epoch in range(epochs):
    lstm_model.train()
    optimizer_lstm.zero_grad()
    out_lstm = lstm_model(X_train_t)
    loss_lstm = criterion(out_lstm, y_train_t)
    loss_lstm.backward()
    optimizer_lstm.step()

    gru_model.train()
    optimizer_gru.zero_grad()
    out_gru = gru_model(X_train_t)
    loss_gru = criterion(out_gru, y_train_t)
    loss_gru.backward()
    optimizer_gru.step()

print("Training Completed!")

In [ ]:
lstm_model.eval()
gru_model.eval()
with torch.no_grad():
    pred_lstm = scaler.inverse_transform(lstm_model(X_test_t).numpy())
    pred_gru = scaler.inverse_transform(gru_model(X_test_t).numpy())
    actual = scaler.inverse_transform(y_test_t.numpy())

plt.figure(figsize=(12, 6))
plt.plot(actual, label='Actual Price', color='black')
plt.plot(pred_lstm, label='LSTM Prediction', color='blue')
plt.plot(pred_gru, label='GRU Prediction', color='orange')
plt.legend()
plt.title('Amazon Stock Price Prediction (LSTM vs GRU)')
plt.show()